# Brain Index — Activation Capture

In [ ]:
!pip install transformers bitsandbytes datasets pandas tqdm torch -q

In [ ]:
import os, sys, ast, time, traceback
from typing import List, Dict, Tuple
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm
import torch.nn.functional as F

# Add scripts directory (parent of utils) to path
sys.path.insert(0, '/kaggle/working/scripts')
from utils.activation_extractor import extract_sparse_neurons, compute_layer_statistics
from utils.sparse_storage import encode_sparse, decode_sparse, verify_encoding
from utils.dataset_builder import build_diverse_dataset

def extract_sparse_neurons(hidden_tensor, top_k=300):
    lt = hidden_tensor[0, -1, :] if hidden_tensor.dim() == 3 else (hidden_tensor[-1, :] if hidden_tensor.dim() == 2 else hidden_tensor)
    activated = F.relu(lt)
    values, indices = activated.topk(k=min(top_k, activated.numel()))
    max_val = values.max().item()
    normalized = ((values / max_val) * 255).round().int() if max_val > 0 else values.new_zeros(len(values), dtype=torch.int)
    return [(idx.item(), val.item()) for idx, val in zip(indices, normalized)]

def encode_sparse(n): return str(n)

def verify_encoding(n):
    if not n: return True
    for i, v in n:
        if not (0 <= i < 4096): raise ValueError(f'idx {i} out of range')
        if not (0 <= v <= 255): raise ValueError(f'val {v} out of range')
    return True

print('[LIBS] OK')

In [ ]:
print(f"[GPU] CUDA: {torch.cuda.is_available()}")

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"
LAYERS = [4, 9, 14, 19, 24, 29, 34, 35]
TOP_K = 300
NUM = 100
OUTPUT = "brain_index/data/verification_results.csv"
print(f"[CONFIG] Model={MODEL_NAME}, Layers={LAYERS}, TopK={TOP_K}")

In [ ]:
print('[MODEL] Loading Qwen3-8B with Q4 quantization...')
t0 = time.time()
qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map='auto', quantization_config=qc, attn_implementation='sdpa')
model.eval()
print(f'[MODEL] Loaded in {time.time()-t0:.1f}s')
print(f'[MODEL] type: {type(model).__name__}')
print(f'[MODEL] layers: {len(model.model.layers)}')

In [ ]:
print(f'[HOOKS] Registering {len(LAYERS)} hooks...')
captured = {}
handles = []
def make_hook(i):
    def h(module, input, output):
        captured[i] = output.detach().cpu()
    return h
for i in LAYERS:
    layer = model.model.layers[i]
    handles.append(layer.register_forward_hook(make_hook(i)))
print(f'[HOOKS] Registered {len(handles)} hooks')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"[TOKENIZER] vocab={tokenizer.vocab_size}")

In [ ]:
print('[TEST] Single forward pass...')
inputs = tokenizer('Hello world', return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model(**inputs)
print(f'[TEST] Captured layers: {sorted(captured.keys())}')
assert set(captured.keys()) == set(LAYERS), f'Hook mismatch! got {set(captured.keys())}'
print('[TEST] SUCCESS: All hooks fired!')
sparse = extract_sparse_neurons(captured[19], TOP_K)
verify_encoding(sparse)
print(f'[SPARSE] Layer 19: {len(sparse)} neurons')

In [ ]:
from datasets import load_dataset
import random

DOMAIN_SOURCES = {
    'reasoning': ('open-thoughts/AgentTrove', None, 'conversations'),
    'creative': ('stingning/ultrachat', None, 'data'),
    'code': ('stingning/ultrachat', None, 'data'),
    'general': ('stingning/ultrachat', None, 'data'),
    'summarization': ('stingning/ultrachat', None, 'data'),
    'translation': ('stingning/ultrachat', None, 'data'),
    'agentic': ('open-thoughts/AgentTrove', None, 'conversations'),
}
FILTER_KW = {
    'reasoning': ['solve','calculate','explain','prove','find','determine','reasoning','logic','math'],
    'creative': ['write','story','poem','script','creative','imagine','invent'],
    'code': ['code','function','python','javascript','programming','implement','algorithm','debug'],
    'translation': ['translate','translation','from english','to spanish','to french'],
    'agentic': ['plan','steps','tool','agent','search','find','look up','research'],
}

def extract_prompt(item, col):
    if col == 'conversations' and col in item:
        for conv in reversed(item[col]):
            if isinstance(conv, dict) and conv.get('role') in ('user','human'):
                v = conv.get('value','')
                if v and isinstance(v, str) and len(v) > 10: return v.strip()
    if col == 'data' and col in item:
        for msg in item[col]:
            if isinstance(msg, str) and len(msg) > 10: return msg.strip()
    for c in [col, 'text', 'instruction', 'prompt', 'input', 'message', 'content']:
        if c and c in item:
            v = item[c]
            if v and isinstance(v, str) and len(v) > 10: return v.strip()
            if isinstance(v, list) and len(v) > 0:
                first = v[0]
                if isinstance(first, dict):
                    val = first.get('content', first.get('text', ''))
                    if val and isinstance(val, str) and len(val) > 10: return val.strip()
    return None

def build_dataset(domain_counts):
    all_prompts = []
    for domain, count in domain_counts.items():
        print(f'[DATASET] {domain}: target={count}')
        if domain not in DOMAIN_SOURCES:
            print(f'[DATASET] WARNING: {domain} unknown'); continue
        ds_name, cfg, text_col = DOMAIN_SOURCES[domain]
        try:
            ds = load_dataset(ds_name, cfg, split='train', streaming=True) if cfg else load_dataset(ds_name, split='train', streaming=True)
            print(f'[DATASET]   -> Loaded {ds_name}')
        except Exception as e:
            print(f'[DATASET]   -> ERROR {e}, fallback to ultrachat')
            try:
                ds = load_dataset('stingning/ultrachat', split='train', streaming=True)
                text_col = 'data'
            except:
                all_prompts.extend([f'Sample {domain} prompt {i}' for i in range(count)])
                continue
        raw = []
        for j, item in enumerate(iter(ds)):
            pt = extract_prompt(item, text_col)
            if pt: raw.append(pt)
            if j >= 9999: break
        print(f'[DATASET]   -> {len(raw)} raw prompts')
        if domain in FILTER_KW:
            before = len(raw)
            raw = [p for p in raw if any(kw in p.lower() for kw in FILTER_KW[domain])]
            print(f'[DATASET]   -> Filtered: {before} -> {len(raw)}')
        if not raw:
            raw = [f'Sample {domain} prompt {i}' for i in range(count)]
        if len(raw) >= count:
            step = max(1, len(raw) // count)
            sampled = raw[::step][:count]
        else:
            shortage = count - len(raw)
            extra = (raw * ((shortage // len(raw)) + 1))[:shortage]
            sampled = raw + extra
        print(f'[DATASET]   -> Sampled {len(sampled)} for {domain}')
        all_prompts.extend(sampled)
    random.shuffle(all_prompts)
    print(f'[DATASET] Total: {len(all_prompts)} prompts')
    return all_prompts

print('[DATASET] build_dataset defined')

In [ ]:
domain_config = {'reasoning': 15, 'creative': 10, 'code': 15, 'general': 25, 'summarization': 10, 'translation': 10, 'agentic': 15}
prompts = build_dataset(domain_config)
print(f'[RUN] Built {len(prompts)} prompts')

In [ ]:
print(f'[RUN] Processing {len(prompts)} prompts...')
results = []
run_start = time.time()
for i, prompt in enumerate(prompts):
    if i % 10 == 0:
        elapsed = time.time() - run_start
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        print(f'[RUN] Progress: {i}/{len(prompts)} ({rate:.2f} prompts/sec)')
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    captured.clear()
    with torch.no_grad():
        outputs = model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
    row = {'prompt_id': i, 'prompt_text': prompt[:100]}
    for li in LAYERS:
        sparse = extract_sparse_neurons(captured[li], TOP_K)
        row[f'layer_{li}'] = encode_sparse(sparse)
    results.append(row)
print(f'[RUN] Done! {len(prompts)} prompts in {time.time()-run_start:.1f}s')

In [ ]:
print('[CSV] Writing...')
os.makedirs('brain_index/data', exist_ok=True)
df = pd.DataFrame(results)
df.to_csv(OUTPUT, index=False)
print(f'[CSV] Wrote {len(df)} rows to {OUTPUT}')
print(f'[CSV] Size: {os.path.getsize(OUTPUT)/1e6:.2f} MB')

In [ ]:
print('[SUMMARY] Activation stats (expected ~200-400 neurons/layer):')
for li in LAYERS:
    col = f'layer_{li}'
    counts = [len(ast.literal_eval(r[col])) for r in results[:10]]
    avg = sum(counts) / len(counts)
    print(f'  Layer {li}: avg={avg:.0f} neurons')

In [ ]:
print('[CLEANUP] Removing hooks...')
for h in handles:
    h.remove()
print('[DONE] Verification complete!')